# 07 — Conclusion et synthèse finale

**Cours** : Introduction à Python — M1 BIDABI, Université Sorbonne Paris Nord  
**Auteurs** : Palla Gueye & Karissy Zeinab Cisse  
**Source** : ILOSTAT — Organisation Internationale du Travail

Ce notebook récapitule **tous les chiffres clés** du projet et formule des réponses explicites à chaque question de recherche. Il produit également une figure de synthèse finale.

**Fichier produit** : `figures/10_synthese.png`

---
## 0. Import et reconstruction des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130

PROJECT_ROOT = Path.cwd().parent
RAW_DIR      = PROJECT_ROOT / 'data' / 'raw'
FIGURES_DIR  = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

COLORS       = {'LOW': '#e07b54', 'MEDIUM': '#f0c05a', 'HIGH': '#4caf82'}
NIVEAUX      = ['LOW', 'MEDIUM', 'HIGH']
CODES        = ['EDU_AGGREGATE_BAS', 'EDU_AGGREGATE_INT', 'EDU_AGGREGATE_ADV']
CODES_NIVEAU = {'EDU_AGGREGATE_BAS': 'LOW', 'EDU_AGGREGATE_INT': 'MEDIUM', 'EDU_AGGREGATE_ADV': 'HIGH'}
ISCED_SUP    = ['EDU_ISCED11_5', 'EDU_ISCED11_6', 'EDU_ISCED11_7', 'EDU_ISCED11_8']
ISCED_LABELS = {
    'EDU_ISCED11_5': 'Bac+2', 'EDU_ISCED11_6': 'Licence',
    'EDU_ISCED11_7': 'Master', 'EDU_ISCED11_8': 'Doctorat',
}
REGIONS = {
    'FRA':'Europe','DEU':'Europe','ITA':'Europe','ESP':'Europe','GBR':'Europe',
    'POL':'Europe','NLD':'Europe','BEL':'Europe','SWE':'Europe','NOR':'Europe',
    'CHE':'Europe','AUT':'Europe','PRT':'Europe','GRC':'Europe','CZE':'Europe',
    'HUN':'Europe','ROU':'Europe','BGR':'Europe','HRV':'Europe','SVK':'Europe',
    'FIN':'Europe','DNK':'Europe','IRL':'Europe','LTU':'Europe','LVA':'Europe',
    'EST':'Europe','SVN':'Europe','RUS':'Europe','UKR':'Europe','SRB':'Europe',
    'USA':'Amériques','CAN':'Amériques','MEX':'Amériques','BRA':'Amériques',
    'ARG':'Amériques','COL':'Amériques','CHL':'Amériques','PER':'Amériques',
    'ECU':'Amériques','BOL':'Amériques','PRY':'Amériques','URY':'Amériques',
    'CHN':'Asie-Pacifique','JPN':'Asie-Pacifique','KOR':'Asie-Pacifique',
    'IND':'Asie-Pacifique','IDN':'Asie-Pacifique','THA':'Asie-Pacifique',
    'AUS':'Asie-Pacifique','NZL':'Asie-Pacifique','SGP':'Asie-Pacifique',
    'ZAF':'Afrique','NGA':'Afrique','ETH':'Afrique','EGY':'Afrique',
    'DZA':'Afrique','MAR':'Afrique','TUN':'Afrique','GHA':'Afrique',
    'SAU':'Moyen-Orient','IRN':'Moyen-Orient','TUR':'Moyen-Orient',
    'ISR':'Moyen-Orient','JOR':'Moyen-Orient','KWT':'Moyen-Orient',
}

emploi  = pd.read_csv(RAW_DIR / 'EMP_TEMP_SEX_EDU_NB_A.csv')
chomage = pd.read_csv(RAW_DIR / 'UNE_TUNE_SEX_EDU_NB_A.csv')

def build(sex_filter, codes=None):
    """Construit le dataset fusionné emploi+chômage avec taux calculé."""
    if codes is None:
        codes = CODES
    e = emploi[
        (emploi['sex'] == sex_filter) & (emploi['classif1'].isin(codes)) &
        emploi['obs_value'].notna() & (emploi['obs_status'] != 'U') &
        (emploi['time'] >= 2000) & (emploi['time'] <= 2023)
    ]
    u = chomage[
        (chomage['sex'] == sex_filter) & (chomage['classif1'].isin(codes)) &
        chomage['obs_value'].notna() & (chomage['obs_status'] != 'U') &
        (chomage['time'] >= 2000) & (chomage['time'] <= 2023)
    ]
    df = pd.merge(
        e[['ref_area','time','classif1','obs_value']].rename(columns={'obs_value':'emp'}),
        u[['ref_area','time','classif1','obs_value']].rename(columns={'obs_value':'une'}),
        on=['ref_area','time','classif1'], how='inner'
    )
    df['taux']   = df['une'] / (df['une'] + df['emp']) * 100
    df['niveau'] = df['classif1'].map(CODES_NIVEAU)
    df['region'] = df['ref_area'].map(REGIONS)
    return df

df       = build('SEX_T')
df_f     = build('SEX_F')
df_m     = build('SEX_M')
df_isced = build('SEX_T', ISCED_SUP)

taux_niv = df.groupby('niveau')['taux'].mean().reindex(NIVEAUX)
taux_f   = df_f.groupby('niveau')['taux'].mean().reindex(NIVEAUX)
taux_m   = df_m.groupby('niveau')['taux'].mean().reindex(NIVEAUX)
taux_sup = df_isced.groupby('classif1')['taux'].mean().reindex(ISCED_SUP)

pivot_paradox = df.groupby(['ref_area','niveau'])['taux'].mean().unstack().dropna()
pivot_paradox['ecart'] = pivot_paradox['HIGH'] - pivot_paradox['LOW']
n_paradox = (pivot_paradox['ecart'] > 0).sum()
n_total   = len(pivot_paradox)

ecart_mondial = df.groupby(['time','niveau'])['taux'].mean().unstack()
ecart_mondial['ecart'] = ecart_mondial['LOW'] - ecart_mondial['HIGH']

taux_reg = (
    df[df['region'].notna()]
    .groupby(['region','niveau'])['taux'].mean()
    .unstack()[['LOW','HIGH']]
)
taux_reg['ecart'] = taux_reg['LOW'] - taux_reg['HIGH']

avant_2008  = df[(df['time']>=2005)&(df['time']<=2007)].groupby('niveau')['taux'].mean().reindex(NIVEAUX)
pic_2008    = df[(df['time']>=2009)&(df['time']<=2010)].groupby('niveau')['taux'].mean().reindex(NIVEAUX)
avant_covid = df[(df['time']>=2017)&(df['time']<=2019)].groupby('niveau')['taux'].mean().reindex(NIVEAUX)
pic_covid   = df[(df['time']>=2020)&(df['time']<=2021)].groupby('niveau')['taux'].mean().reindex(NIVEAUX)
hausse_2008  = ((pic_2008  - avant_2008)  / avant_2008  * 100).reindex(NIVEAUX)
hausse_covid = ((pic_covid - avant_covid) / avant_covid * 100).reindex(NIVEAUX)

print('Données prêtes.')
print(f'  Pays analysés : {df["ref_area"].nunique()}')
print(f'  Période       : {df["time"].min()}–{df["time"].max()}')

---
## 1. Figure de synthèse — résumé visuel en 3 panneaux

In [ ]:
couleurs_sup = ['#a8d8c8', '#4caf82', '#2e7d5e', '#1a4d3a']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panneau 1 : taux par niveau
ax = axes[0]
bars = ax.bar(['Faible', 'Interméd.', 'Élevé'], taux_niv.values,
              color=[COLORS[k] for k in NIVEAUX], edgecolor='white', linewidth=1.5, width=0.5)
for b, v in zip(bars, taux_niv.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
            f'{v:.1f}%', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Taux de chômage\npar niveau de diplôme', fontweight='bold', fontsize=11)
ax.set_ylabel('Taux moyen (%)'); ax.set_ylim(0, taux_niv.max()*1.3)

# Panneau 2 : ISCED 5–8
ax = axes[1]
bars = ax.bar([ISCED_LABELS[c] for c in ISCED_SUP], taux_sup.values,
              color=couleurs_sup, edgecolor='white', linewidth=1.5, width=0.5)
for b, v in zip(bars, taux_sup.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05,
            f'{v:.1f}%', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Taux de chômage\npar diplôme universitaire', fontweight='bold', fontsize=11)
ax.set_ylabel('Taux moyen (%)'); ax.set_ylim(0, taux_sup.max()*1.35)
ax.tick_params(axis='x', labelsize=9)

# Panneau 3 : genre
ax = axes[2]
x = np.arange(3); w = 0.32
bars_f = ax.bar(x-w/2, taux_f.values, w, label='Femmes', color='#d4869c', edgecolor='white')
bars_m = ax.bar(x+w/2, taux_m.values, w, label='Hommes', color='#6a9ec5', edgecolor='white')
for b, v in zip(bars_f, taux_f.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
            f'{v:.1f}%', ha='center', fontsize=9, color='#c06080', fontweight='bold')
for b, v in zip(bars_m, taux_m.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
            f'{v:.1f}%', ha='center', fontsize=9, color='#3a7aaa', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(['Faible', 'Interméd.', 'Élevé'], fontsize=9)
ax.set_title('Taux de chômage\npar niveau et par genre', fontweight='bold', fontsize=11)
ax.set_ylabel('Taux moyen (%)'); ax.legend(fontsize=10)
ax.set_ylim(0, max(taux_f.max(), taux_m.max())*1.3)

plt.suptitle('Synthèse — Éducation, genre et chômage dans le monde (2000–2023)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '10_synthese.png', bbox_inches='tight')
plt.show()
print('Figure 10 sauvegardée.')

---
## 2. Réponses aux questions de recherche

In [ ]:
sep = '=' * 70

print(sep)
print('PROBLÉMATIQUE PRINCIPALE')
print('Dans quelle mesure le niveau de diplôme influence-t-il l\'accès à l\'emploi ?')
print(sep)
print()
print(f'Taux de chômage moyen mondial (2000–2023) :')
print(f'  Faible (LOW)          : {taux_niv["LOW"]:.1f}%')
print(f'  Intermédiaire (MEDIUM): {taux_niv["MEDIUM"]:.1f}%')
print(f'  Élevé (HIGH)          : {taux_niv["HIGH"]:.1f}%')
print(f'  Écart LOW → HIGH      : {taux_niv["LOW"] - taux_niv["HIGH"]:.1f} points de %')
print()
print('→ Le diplôme influence significativement l\'accès à l\'emploi. Un diplôme élevé')
print('  est associé à un taux de chômage nettement plus faible au niveau mondial.')
print('  Cet effet est persistant dans le temps et résiste aux chocs économiques.')

print()
print(sep)
print('SOUS-QUESTION 1 — Le diplôme protège-t-il pareil selon la région ?')
print(sep)
print()
print('Taux de chômage et écart LOW–HIGH par région :')
display(taux_reg.round(2).sort_values('ecart', ascending=False))
print()
region_max = taux_reg['ecart'].idxmax()
region_min = taux_reg['ecart'].idxmin()
print(f'→ NON. L\'effet est universel mais son intensité varie fortement.')
print(f'  Protection la plus forte : {region_max} ({taux_reg.loc[region_max,"ecart"]:.1f} pts)')
print(f'  Protection la plus faible : {region_min} ({taux_reg.loc[region_min,"ecart"]:.1f} pts)')

print()
print(sep)
print('SOUS-QUESTION 2 — Les diplômes universitaires ont-ils tous la même valeur ?')
print(sep)
print()
print('Taux de chômage moyen par niveau universitaire :')
for code in ISCED_SUP:
    print(f'  {ISCED_LABELS[code]:10s} : {taux_sup[code]:.1f}%')
print()
print(f'→ NON. Le taux diminue du Bac+2 ({taux_sup["EDU_ISCED11_5"]:.1f}%) au Doctorat ({taux_sup["EDU_ISCED11_8"]:.1f}%).')
print(f'  Écart Bac+2 → Doctorat : {taux_sup["EDU_ISCED11_5"] - taux_sup["EDU_ISCED11_8"]:.1f} pts')
print('  Nuance : la Licence affiche un taux supérieur au Bac+2 en moyenne mondiale')
print('  (phénomène plus marqué en Europe, et qui s\'estompe depuis 2018).')

print()
print(sep)
print('SOUS-QUESTION 3 — Tous les diplômes protègent-ils du chômage pareil ?')
print(sep)
print()
print(f'→ NON. Chaque niveau supplémentaire réduit le taux de chômage.')
print(f'  Écart global LOW–HIGH : {taux_niv["LOW"] - taux_niv["HIGH"]:.1f} pts au niveau mondial.')

---
## 3. Résultats contre-intuitifs (pour la présentation)

In [ ]:
print(sep)
print('RÉSULTATS CONTRE-INTUITIFS')
print(sep)
print()

print('1. Le paradoxe du chômage éduqué')
print(f'   Dans {n_paradox} pays sur {n_total} ({n_paradox/n_total*100:.0f}%), les très diplômés')
print('   chôment davantage que les peu diplômés (Cameroun, Inde, Tunisie...)')
print('   → Le marché formel est trop petit pour absorber tous les diplômés.')
print()

print('2. Ce phénomène s\'aggrave dans le temps')
print('   2000–2003 : ~30% des pays paradoxaux → 2020–2023 : ~48% des pays.')
print('   → La massification du supérieur dépasse la croissance du marché formel.')
print()

print('3. COVID vs crise 2008 : deux crises, deux réalités')
print(f'   Hausse relative 2008 — LOW: {hausse_2008["LOW"]:+.1f}% | HIGH: {hausse_2008["HIGH"]:+.1f}%')
print(f'   Hausse relative COVID — LOW: {hausse_covid["LOW"]:+.1f}% | HIGH: {hausse_covid["HIGH"]:+.1f}%')
print('   → En 2008 les diplômés souffrent plus. En 2020 le télétravail les protège.')
print()

print('4. L\'avantage du diplôme se réduit de moitié en 20 ans')
print(f'   Écart LOW–HIGH en 2000 : {ecart_mondial.loc[2000,"ecart"]:.1f} pts')
print(f'   Écart LOW–HIGH en 2019 : {ecart_mondial.loc[2019,"ecart"]:.1f} pt')
print('   → La massification du supérieur réduit l\'avantage comparatif du diplôme.')
print()

print('5. Genre : le diplôme réduit l\'écart mais ne l\'efface pas')
for niv in NIVEAUX:
    e = taux_f[niv] - taux_m[niv]
    print(f'   {niv:8s} — Femmes: {taux_f[niv]:.1f}% | Hommes: {taux_m[niv]:.1f}% | Écart: +{e:.1f} pts')

---
## 4. Limites de l'analyse

In [ ]:
print(sep)
print('LIMITES DE L\'ANALYSE')
print(sep)
print()

print('1. TAUX CALCULÉ, NON OFFICIEL')
print('   taux = nb_chomeurs / (nb_employes + nb_chomeurs) × 100')
print('   C\'est une approximation — la force de travail totale inclut aussi les inactifs.')
print()

print('2. EMPLOI INFORMEL NON CAPTURÉ')
print('   Les données ILO mesurent l\'emploi formel déclaré. Dans les pays en développement,')
print('   l\'emploi informel (agriculture, artisanat) n\'apparaît pas. Cela explique')
print('   pourquoi des pays ont des taux LOW très faibles (≠ absence de chômage).')
print()

print('3. COUVERTURE GÉOGRAPHIQUE VARIABLE')
print('   Tous les pays ne reportent pas leurs données chaque année.')
print('   Les pays avec de bonnes statistiques ne sont pas représentatifs de tous les pays.')
print()

print('4. RUPTURES DE SÉRIE (obs_status = B) CONSERVÉES')
print('   On a exclu les données peu fiables (U) mais conservé les ruptures de série (B).')
print('   Cela peut introduire de légères discontinuités dans certains pays.')
print()

print('5. CORRÉLATION ≠ CAUSALITÉ')
print('   On montre que diplôme et taux de chômage sont corrélés.')
print('   On ne prouve pas que le diplôme CAUSE une meilleure insertion — d\'autres')
print('   facteurs jouent un rôle (réseau social, secteur, origine géographique).')
print()

print('6. DONNÉES 2024–2025 NON UTILISÉES')
print('   95 pays ont des données 2024 disponibles dans ILO. On s\'est arrêté à 2023')
print('   pour garantir la cohérence et la complétude sur toute la période analysée.')

---
## 5. Statistiques descriptives complètes (utiles pour le rapport)

In [ ]:
print(sep)
print('STATISTIQUES DESCRIPTIVES — À INTÉGRER DANS LE RAPPORT')
print(sep)
print()

print('Description des bases de données brutes :')
print(f'  Emploi  : {len(emploi):,} lignes | {emploi["ref_area"].nunique()} pays | {emploi["time"].min()}–{emploi["time"].max()}')
print(f'  Chômage : {len(chomage):,} lignes | {chomage["ref_area"].nunique()} pays | {chomage["time"].min()}–{chomage["time"].max()}')
print()

print('Après nettoyage et fusion :')
print(f'  Pays dans l\'analyse finale : {df["ref_area"].nunique()}')
print(f'  Observations dans le dataset fusionné : {len(df):,}')
print(f'  Période retenue : 2000–2023 (24 années)')
print()

print('Distribution taux de chômage (toutes années, tous pays) :')
display(df.groupby('niveau')['taux'].describe().round(2).reindex(['LOW','MEDIUM','HIGH']))
print()

print('Valeurs extrêmes notables :')
max_low  = df[df['niveau']=='LOW'].nlargest(1,'taux')
max_high = df[df['niveau']=='HIGH'].nlargest(1,'taux')
print(f'  Taux max LOW  : {max_low["taux"].values[0]:.1f}% ({max_low["ref_area"].values[0]}, {int(max_low["time"].values[0])})')
print(f'  Taux max HIGH : {max_high["taux"].values[0]:.1f}% ({max_high["ref_area"].values[0]}, {int(max_high["time"].values[0])})')
print('  → Ces valeurs extrêmes sont mentionnées comme limites dans le rapport.')